In [1]:
import os
import sys
import tifffile
import pandas as pd
import concurrent.futures
from tqdm.notebook import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../src')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../src/tools')))
from utils import estimate_snr_acldr
from _2_data_loading_helpers import parse_data_structure


DATA_ROOT = "../trinity_sem_dataset"
SNR_ESTIMATIONS_CSV = "snr_estimations.csv"


def process_datasets(root_dir):
    parsed_triplets = parse_data_structure(root_dir)
    
    measurements = []
    
    def process_detector(triplet_id, detector, fpath):
        img = tifffile.imread(fpath)
        return {"triplet_ID": triplet_id, "detector": detector, "SNR": estimate_snr_acldr(img, autoregressive_order=4)}
        
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for triplet in parsed_triplets:
            for detector in ['ETD', 'T1', 'T2']:
                futures.append(executor.submit(process_detector, triplet['ID'], detector, triplet[detector]))
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Calculating SNR (per detector)"):
            measurements.append(future.result())
                
    df = pd.DataFrame(measurements)
    df.sort_values(by=["triplet_ID", "detector"], inplace=True)
    df.to_csv(SNR_ESTIMATIONS_CSV, index=False)
    print(f"saved {len(df)} records to {SNR_ESTIMATIONS_CSV}")

process_datasets(DATA_ROOT)

Calculating SNR (per detector):   0%|          | 0/504 [00:00<?, ?it/s]

saved 504 records to snr_estimations.csv
